In [ ]:
import os
import cv2
import json
from pathlib import Path
import torch
import shutil

### Create a "blacklist" of images in the uncertain folder from Omishas dataset

In [ ]:
uncertain_dir = Path("validity_datasetv1/uncertain")
blacklist = {f.name for f in uncertain_dir.glob("*.jpg")}

In [ ]:
print(len(blacklist))

### Create validity dataset using Omishas dataset
- Run 1: 1.10 crop_buff -> validity_datasetv1
- Run 2: 1.05 crop_buff -> validity_datasetv2

In [ ]:
def create_validity_dataset(image_path, json_data, output_dir, crop_buff=1.05):
    rect_img = cv2.imread(str(image_path))
    h, w, _ = rect_img.shape
    print(h, w)
    rows = json_data["rows"]
    cols = json_data["cols"]
    print(json_data["warp_size"])
    cell_h = h // rows
    cell_w = w // cols
    labels = json_data["cells"]
    label_map = json_data["label_map"]
    for i in range(rows):
        for j in range(cols):
            idx = i * cols + j
            # get label from json
            label_val = labels[idx]
            label_name = label_map[str(label_val)]
            # crop logic
            # get center of cell for buffer
            cx = (j + 0.5) * cell_w
            cy = (i + 0.5) * cell_h
            # apply buffer
            buff_w = cell_w * crop_buff
            buff_h = cell_h * crop_buff
            # calculate buffered coords
            y1 = int(max(0, cy - buff_h / 2))
            y2 = int(min(h, cy + buff_h / 2))
            x1 = int(max(0, cx - buff_w / 2))
            x2 = int(min(w, cx + buff_w / 2))
            cell_crop = rect_img[y1:y2, x1:x2]
            save_path = f"{output_dir}/{label_name}/{json_data['image_name']}_cell_{idx}.jpg"
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            cv2.imwrite(save_path, cell_crop)


In [ ]:
base_path = Path("Labeled Data")
rectified_path = base_path / "Rectified"
output_path = Path("validity_datasetv2")
for json_file in base_path.glob("*.json"):
    # lod json
    with open(json_file, 'r') as f:
        json_data = json.load(f)
    target_img_name = f"{json_data['image_name']}.rectified.jpg"
    target_img_path = rectified_path / target_img_name
    if target_img_path.exists():
        print(f"Processing {target_img_name}")
        create_validity_dataset(target_img_path, json_data, output_path)
    else:
        print(f"No image found: {json_file.name}")

In [ ]:
# Check one from v1 blacklist
v1_sample = next(iter(blacklist))

# Check one from v2 folder
v2_sample = next(Path("validity_datasetv2/occupied").glob("*.jpg")).name

print(f"V1 Name: '{v1_sample}'")
print(f"V2 Name: '{v2_sample}'")
print(f"Are they identical? {v1_sample == v2_sample}")

#### Move blacklisted cell images to uncertain

In [ ]:
uncertain_v1 = Path("validity_datasetv1/uncertain")
v2_base = Path("validity_datasetv2")
uncertain_v2 = v2_base / "uncertainv2"
uncertain_v2.mkdir(parents=True, exist_ok=True)

blacklist = {f.name.strip() for f in uncertain_v1.glob("*.jpg")}
print(f"Items in blacklist: {len(blacklist)}")

move_count = 0
for label in ["empty", "occupied"]:
    curr_dir = v2_base / label
    if not curr_dir.exists():
        continue
        
    for img_p in curr_dir.glob("*.jpg"):
        # We strip the name here too to ensure a clean comparison
        clean_name = img_p.name.strip()
        
        if clean_name in blacklist:
            dest = uncertain_v2 / img_p.name
            shutil.move(str(img_p), str(dest))
            move_count += 1

print(f"Moved {move_count} images to {uncertain_v2}")

In [ ]:
from pathlib import Path

def print_counts(version="v2"):
    base = Path(f"validity_dataset{version}")
    print(f"--- Counts for {version} ---")
    for folder in ["empty", "occupied", "uncertain"]:
        path = base / folder
        if path.exists():
            count = len(list(path.glob("*.jpg")))
            print(f"{folder}: {count} images")
        else:
            print(f"{folder}: Folder not found")
    print("-" * 20)

print_counts("v1")
print_counts("v2")

### Training and inference for v1 dataset

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, random_split
from collections import Counter

# Added augmentations to help with the 95% threshold
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
# test set - no augmentations
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Toggle this for v1 vs v2
DATASET_ROOT = 'validity_datasetv1' 
train_data = datasets.ImageFolder(root=DATASET_ROOT, transform=train_transforms)
test_data = datasets.ImageFolder(root=DATASET_ROOT, transform=eval_transforms)
class_to_idx = train_data.class_to_idx
indices = [i for i, (_, label) in enumerate(train_data.samples) if label in [class_to_idx['empty'], class_to_idx['occupied']]]

total_size = len(indices)
train_idx, val_idx, test_idx = random_split(range(total_size), [0.8, 0.1, 0.1])
train_set = Subset(train_data, [indices[i] for i in train_idx])
val_set = Subset(test_data, [indices[i] for i in val_idx])
test_set = Subset(test_data, [indices[i] for i in test_idx])
# Check classes
print(f"Total images in folder: {len(train_data)}")
print(f"Images kept (Empty + Occupied): {len(indices)}")

# Pull labels using the filtered indices
labels = [train_data.targets[i] for i in indices]
distribution = Counter(labels)

# Map the numbers back to names for a clearer print
print(f"Class Distribution: { {train_data.classes[k]: v for k, v in distribution.items()} }")

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)
test_loader = DataLoader(test_set, batch_size=16)

# Model setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
print(f"Dataset: {DATASET_ROOT} | Train: {len(train_set)} | Test: {len(test_set)} | Val: {len(val_set)}")

# Training loop
for epoch in range(12): # 12 epochs
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(); outputs = model(inputs)
        loss = criterion(outputs, labels); loss.backward(); optimizer.step()
        running_loss += loss.item()
    
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0); correct += (preds == labels).sum().item()
            
    print(f"Epoch {epoch+1} - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {100*correct/total:.2f}%")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()
mismatches = []
all_preds, all_labels = [], []
class_names = ["empty", "occupied"]

# run inference
with torch.no_grad():
    for i in range(len(test_set)):
        img_tensor, label = test_set[i]
        input_tensor = img_tensor.unsqueeze(0).to(device)
        output = model(input_tensor)
        pred = torch.max(output, 1)[1].item()

        all_preds.append(pred)
        all_labels.append(label)

        if pred != label:
            mismatches.append({
                'image': img_tensor, 
                'actual': class_names[label], 
                'pred': class_names[pred]
            })

all_preds, all_labels = np.array(all_preds), np.array(all_labels)

# confusion matrix if seaborn decides to work :)
plt.figure(figsize=(8, 6))
matrix = np.zeros((2, 2), dtype=int)
for a, p in zip(all_labels, all_preds):
    matrix[a, p] += 1

sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Sapling Validity')
plt.show()

tn, fp, fn, tp = matrix.ravel()
accuracy = (tp + tn) / len(all_labels)
print(f"--- Test Results ---\nAccuracy: {accuracy:.2%}\n" + "-"*20)

In [ ]:
# display misclassifications
num_to_show = min(len(mismatches), 12)
if num_to_show > 0:
    print(f"Displaying {num_to_show} of {len(mismatches)} mismatches:")
    plt.figure(figsize=(15, 8))
    for i in range(num_to_show):
        m = mismatches[i]
        # Un-normalize
        img = m['image'].permute(1, 2, 0).cpu().numpy()
        img = np.clip(img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)
        
        plt.subplot(3, 4, i + 1)
        plt.imshow(img)
        plt.title(f"ACT: {m['actual']} | PRED: {m['pred']}", color='red', fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
model_path = "sapling_validity_resnet18.pt"
# torch.save(model.state_dict(), model_path)

Loading model for inference

In [ ]:
import torch
from torchvision import models
import torch.nn as nn

model = models.resnet18()
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.load_state_dict(torch.load("sapling_validity_resnet18.pt", map_location=device))
model.to(device)
model.eval()

### Training and inference for v2 dataset

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, random_split
from collections import Counter

# Added augmentations to help with the 95% threshold
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
# test set - no augmentations
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Toggle this for v1 vs v2
DATASET_ROOT = 'validity_datasetv2' 
train_data = datasets.ImageFolder(root=DATASET_ROOT, transform=train_transforms)
test_data = datasets.ImageFolder(root=DATASET_ROOT, transform=eval_transforms)
class_to_idx = train_data.class_to_idx
indices = [i for i, (_, label) in enumerate(train_data.samples) if label in [class_to_idx['empty'], class_to_idx['occupied']]]

total_size = len(indices)
train_idx, val_idx, test_idx = random_split(range(total_size), [0.8, 0.1, 0.1])
train_set = Subset(train_data, [indices[i] for i in train_idx])
val_set = Subset(test_data, [indices[i] for i in val_idx])
test_set = Subset(test_data, [indices[i] for i in test_idx])
# Check classes
print(f"Total images in folder: {len(train_data)}")
print(f"Images kept (Empty + Occupied): {len(indices)}")

# Pull labels using the filtered indices
labels = [train_data.targets[i] for i in indices]
distribution = Counter(labels)

# Map the numbers back to names for a clearer print
print(f"Class Distribution: { {train_data.classes[k]: v for k, v in distribution.items()} }")

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)
test_loader = DataLoader(test_set, batch_size=16)

# Model setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
print(f"Dataset: {DATASET_ROOT} | Train: {len(train_set)} | Test: {len(test_set)} | Val: {len(val_set)}")

# Training loop
for epoch in range(12): # 12 epochs
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(); outputs = model(inputs)
        loss = criterion(outputs, labels); loss.backward(); optimizer.step()
        running_loss += loss.item()
    
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0); correct += (preds == labels).sum().item()
            
    print(f"Epoch {epoch+1} - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {100*correct/total:.2f}%")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()
mismatches = []
all_preds, all_labels = [], []
class_names = ["empty", "occupied"]

# run inference
with torch.no_grad():
    for i in range(len(test_set)):
        img_tensor, label = test_set[i]
        input_tensor = img_tensor.unsqueeze(0).to(device)
        output = model(input_tensor)
        pred = torch.max(output, 1)[1].item()

        all_preds.append(pred)
        all_labels.append(label)

        if pred != label:
            mismatches.append({
                'image': img_tensor, 
                'actual': class_names[label], 
                'pred': class_names[pred]
            })

all_preds, all_labels = np.array(all_preds), np.array(all_labels)

# confusion matrix if seaborn decides to work :)
plt.figure(figsize=(8, 6))
matrix = np.zeros((2, 2), dtype=int)
for a, p in zip(all_labels, all_preds):
    matrix[a, p] += 1

sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Sapling Validity')
plt.show()

tn, fp, fn, tp = matrix.ravel()
accuracy = (tp + tn) / len(all_labels)
print(f"--- Test Results ---\nAccuracy: {accuracy:.2%}\n" + "-"*20)

In [ ]:
# display misclassifications
num_to_show = min(len(mismatches), 12)
if num_to_show > 0:
    print(f"Displaying {num_to_show} of {len(mismatches)} mismatches:")
    plt.figure(figsize=(15, 8))
    for i in range(num_to_show):
        m = mismatches[i]
        # Un-normalize
        img = m['image'].permute(1, 2, 0).cpu().numpy()
        img = np.clip(img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)
        
        plt.subplot(3, 4, i + 1)
        plt.imshow(img)
        plt.title(f"ACT: {m['actual']} | PRED: {m['pred']}", color='red', fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Demo 3/16

In [ ]:
from IPython.display import Image, display

rect_seg_img = "../data/Labeled Data/Debug/tray_0116.jpg.rectified.debug.jpg"
rect_img = "../data/Labeled Data/tray_0116.jpg"

display(Image(filename=rect_img))

In [ ]:
display(Image(filename=rect_seg_img))

In [ ]:
for i in range(21):
    display(Image(filename=f"../data/validity_datasetv1/demo/tray_0116.jpg_cell_{i}.jpg"))

In [ ]:
import torch
from torchvision import models
import torch.nn as nn
from pathlib import Path
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F

model = models.resnet18()
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.load_state_dict(torch.load("../models/sapling_validity_resnet18.pt", map_location=device))
model.to(device)
model.eval()

inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# path to  demo images
demo_path = Path("../data/validity_datasetv1/demo")
cell_files = sorted(list(demo_path.glob("*.jpg")), 
                    key=lambda x: int(x.stem.split('_')[-1])) 
class_names = ['empty', 'occupied'] 

results = []

for img_path in cell_files:
    # load and transform
    img = Image.open(img_path).convert('RGB')
    print(img_path)
    input_tensor = inference_transforms(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)
        
        # Apply softmaxc
        probs = F.softmax(logits, dim=1)
        
        # Get the top class and its probability
        conf, pred = torch.max(probs, 1)
        
        results.append({
            'cell_id': img_path.name,
            'prediction': class_names[pred.item()],
            'confidence': conf.item()
        })

# Preview the first 5 results
for r in results[:5]:
    print(f"{r['cell_id']}: {r['prediction']} ({r['confidence']:.2%})")

In [ ]:
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- 1. LOAD ASSETS ---
tray_id = "tray_0116"
rect_img_path = f"../data/Labeled Data/Rectified/{tray_id}.jpg.rectified.jpg"
json_path = f"../data/Labeled Data/{tray_id}.jpg.labels.json"

# Load the image and the grid metadata
image = cv2.imread(rect_img_path)
with open(json_path, 'r') as f:
    grid_data = json.load(f)

# Grid info
rows = grid_data["rows"]
cols = grid_data["cols"]
h, w, _ = image.shape
cell_h = h // rows
cell_w = w // cols

font = cv2.FONT_HERSHEY_DUPLEX 
font_scale = 0.5  
thickness = 1     
box_thickness = 2

viz_img = image.copy()

# For Tray-Level Stats
total_cells = len(results)
occupied_count = sum(1 for data in results if data['prediction'].lower() == 'occupied')
viability_pct = (occupied_count / total_cells) * 100 if total_cells > 0 else 0

for data in results:
    cell_idx = int(data['cell_id'].split('_cell_')[-1].split('.')[0])
    
    r = cell_idx // cols
    c = cell_idx % cols
    
    # Using float math to prevent pixel drift
    x1 = int(c * (w / cols))
    y1 = int(r * (h / rows))
    x2 = int((c + 1) * (w / cols))
    y2 = int((r + 1) * (h / rows))
    
    # Logic: Green for Occupied, Red for Empty
    color = (0, 255, 0) if data['prediction'].lower() == 'occupied' else (0, 0, 255)
    text_color = (255, 255, 255)
    
    conf_val = data['confidence'] * 100
    label = f"{data['prediction'][:1].upper()}: {conf_val:.2f}%"

    cv2.rectangle(viz_img, (x1, y1), (x2, y2), color, box_thickness)
    
    (t_w, t_h), baseline = cv2.getTextSize(label, font, font_scale, thickness)
    cv2.rectangle(viz_img, (x1, y1), (x1 + t_w + 6, y1 + t_h + 10), color, -1)
    
    # Draw Text
    cv2.putText(viz_img, label, (x1 + 3, y1 + t_h + 4), 
                font, font_scale, text_color, thickness, cv2.LINE_AA) 


plt.figure(figsize=(20, 14))
plt.imshow(cv2.cvtColor(viz_img, cv2.COLOR_BGR2RGB))
plt.title(f"Tray ID: {tray_id} | Total Viability: {viability_pct:.2f}%", fontsize=20, fontweight='bold')
plt.axis('off')
plt.show()

# cv2.imwrite(f"{tray_id}_demo_output.jpg", viz_img)